# เทรนตัวตรวจจับกล่องสลิปเอง (RF-DETR Small) จาก label ที่แก้แล้ว

แทนการเรียก Roboflow API: เทรนบน Kaggle แล้วเอาน้ำหนัก/ONNX ไปรันเองทุกที่ ไม่มีโควตา
- รูป: ใช้ Kaggle Dataset ที่อัปโหลดสลิปไว้แล้ว (แก้ `IMAGE_DIR`) ไม่ต้องอัปโหลดรูปใหม่
- label: ฝังในเซลล์ถัดไป (คีย์ = md5 ของไฟล์รูป จึงไม่ขึ้นกับชื่อไฟล์/โฟลเดอร์ที่ Kaggle เปลี่ยน)
- แบ่ง train/valid/test ตามที่ Roboflow แบ่งไว้เดิม -> ตัวเลขบน valid/test เป็นของรูปที่โมเดลไม่เคยเห็นในรอบเทรนนี้
- ตั้ง Accelerator = GPU (T4) และเปิด Internet (โหลด `rfdetr` กับน้ำหนักเริ่มต้น)

In [ ]:
!pip -q install rfdetr

In [ ]:
import base64, gzip, hashlib, json, random, shutil
from collections import Counter
from pathlib import Path

IMAGE_DIR = Path("/kaggle/input/datasets/chujikgch/887uu7")     # <- โฟลเดอร์ dataset รูปสลิป (rglob เข้าโฟลเดอร์ย่อยทุกชั้น)
WORK = Path("/kaggle/working/slip_coco")
EPOCHS = 50
BATCH = 4
GRAD_ACCUM = 4
CLASSES = ["bank", "memo", "money", "name", "sent", "silp", "type payment"]
print("พร้อม")

In [ ]:
# label ที่แก้แล้ว (260 รูป) คีย์ = md5 ของไฟล์รูป ค่า = split, ขนาดรูป, กล่อง [class, x0, y0, x1, y1] (พิกเซล)
_LABELS_B64 = "..."
LABELS = json.loads(gzip.decompress(base64.b64decode(_LABELS_B64)))
print("label", len(LABELS), "รูป |", Counter(v["split"] for v in LABELS.values()))

In [ ]:
# สร้าง dataset รูปแบบ COCO (โครงเดียวกับที่ Roboflow export) จากรูปใน IMAGE_DIR + label ด้านบน
if WORK.exists():
    shutil.rmtree(WORK)
SPLITS = {"train": "train", "valid": "valid", "test": "test"}
coco = {s: {"images": [], "annotations": [], "categories": [{"id": 0, "name": "slip-objects", "supercategory": "none"}] +
            [{"id": i + 1, "name": n, "supercategory": "slip-objects"} for i, n in enumerate(CLASSES)]} for s in SPLITS}
for s in SPLITS:
    (WORK / s).mkdir(parents=True)
ann_id, found, unmatched = 1, 0, []
files = sorted(p for p in IMAGE_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
for p in files:
    v = LABELS.get(hashlib.md5(p.read_bytes()).hexdigest())
    if v is None:
        unmatched.append(p.name)
        continue
    found += 1
    s = v["split"]
    name = "img_%04d.jpg" % found
    shutil.copy(p, WORK / s / name)
    img_id = len(coco[s]["images"]) + 1
    coco[s]["images"].append({"id": img_id, "file_name": name, "width": v["w"], "height": v["h"]})
    for cls, x0, y0, x1, y1 in v["b"]:
        coco[s]["annotations"].append({"id": ann_id, "image_id": img_id, "category_id": CLASSES.index(cls) + 1,
                                       "bbox": [x0, y0, x1 - x0, y1 - y0], "area": (x1 - x0) * (y1 - y0), "iscrowd": 0})
        ann_id += 1
for s in SPLITS:
    (WORK / s / "_annotations.coco.json").write_text(json.dumps(coco[s]), encoding="utf-8")
print("ผูกกับ label ได้ %d รูป | ไม่พบ label %d รูป %s" % (found, len(unmatched), unmatched[:3]))
for s in SPLITS:
    print("%-6s รูป %3d | กล่อง %4d" % (s, len(coco[s]["images"]), len(coco[s]["annotations"])))

In [ ]:
from rfdetr import RFDETRSmall

model = RFDETRSmall()
model.train(dataset_dir=str(WORK), epochs=EPOCHS, batch_size=BATCH, grad_accum_steps=GRAD_ACCUM, lr=1e-4,
            output_dir="/kaggle/working/rfdetr_out")

In [ ]:
# ตรวจบนรูปใน valid + test (ที่โมเดลไม่เคยเห็นตอนเทรน) ด้วยเกณฑ์ที่ pipeline ใช้จริง: ต้องเจอกล่อง money / name / sent ครบ
from PIL import Image

THRESH = 0.40
NEED = ["money", "name", "sent"]
names = {int(k): v for k, v in getattr(model, "class_names", {}).items()} if hasattr(model, "class_names") else {}


def cls_name(cid):
    return names.get(int(cid)) or CLASSES[int(cid) - 1]


def predict_boxes(path, thresh=THRESH):
    det = model.predict(Image.open(path).convert("RGB"), threshold=thresh)
    return [(cls_name(c), float(s), [float(v) for v in xy]) for xy, c, s in zip(det.xyxy, det.class_id, det.confidence)]


rows = []
for p in files:
    v = LABELS.get(hashlib.md5(p.read_bytes()).hexdigest())
    if v is None or v["split"] == "train":
        continue
    got = {c for c, _s, _b in predict_boxes(p)}
    want = {b[0] for b in v["b"]} & set(NEED)
    rows.append({"split": v["split"], "file": p.name, "ขาด": sorted(want - got)})
miss = [r for r in rows if r["ขาด"]]
print("รูป valid+test %d | เจอครบทุกช่องที่เฉลยมี %d | ขาดช่อง %d" % (len(rows), len(rows) - len(miss), len(miss)))
for r in miss:
    print(r)

In [ ]:
# เซฟน้ำหนัก + แปลงเป็น ONNX ไว้ใช้ในแอป/เซิร์ฟเวอร์ (ไฟล์อยู่ใน /kaggle/working -> ดาวน์โหลดจากแท็บ Output)
import os
try:
    model.export(output_dir="/kaggle/working/rfdetr_out")
except Exception as err:
    print("export ONNX ไม่สำเร็จ:", repr(err)[:300])
for p in sorted(Path("/kaggle/working/rfdetr_out").rglob("*")):
    if p.is_file() and p.suffix in {".pth", ".onnx", ".pt"}:
        print("%-60s %.1f MB" % (p, os.path.getsize(p) / 1e6))

In [ ]:
# สร้างแคชตรวจจับกล่องของโมเดลที่เทรนเอง ทั้ง 260 รูป (รูปแบบเดียวกับ DETECT_CACHE ใน notebook OCR) -> วางแทนเซลล์แคชเดิมได้เลย
# หมายเหตุ: รูปใน train โมเดลเคยเรียนแล้ว ผล OCR ที่ได้จึงสูงเกินจริงเหมือนเดิม
cache = {}
for p in files:
    key = hashlib.md5(p.read_bytes()).hexdigest()
    v = LABELS.get(key)
    if v is None:
        continue
    cache[key] = {"w": v["w"], "h": v["h"],
                  "p": [[c, round(s, 4), round((b[0] + b[2]) / 2, 1), round((b[1] + b[3]) / 2, 1), round(b[2] - b[0], 1), round(b[3] - b[1], 1)]
                        for c, s, b in predict_boxes(p, 0.2)]}
blob = base64.b64encode(gzip.compress(json.dumps(cache, separators=(",", ":")).encode(), 9)).decode()
Path("/kaggle/working/detect_cache_own.b64.txt").write_text(blob)
print("แคช %d รูป | %d ตัวอักษร -> /kaggle/working/detect_cache_own.b64.txt" % (len(cache), len(blob)))